# Module B - Query Processing on Google Colab

This notebook tests the query processing pipeline (language detection + translation).

## Steps:
1. Install dependencies (transformers, torch, langdetect)
2. Copy query processing code
3. Run tests

In [ ]:
# Step 1: Install dependencies
!pip install -q transformers torch langdetect
print("✅ Dependencies installed")

In [ ]:
# Step 2: Language Detector
from langdetect import detect, LangDetectException

def detect_language(query: str) -> str:
    """Detect if query is in English or Bangla."""
    if not query or not query.strip():
        return 'unknown'
    
    try:
        lang = detect(query)
        
        if lang == 'bn':
            return 'bn'
        elif lang in ['en']:
            return 'en'
        else:
            bengali_chars = sum(1 for c in query if '\u0980' <= c <= '\u09FF')
            if bengali_chars > len(query) * 0.5:
                return 'bn'
            return 'en'
            
    except LangDetectException:
        bengali_chars = sum(1 for c in query if '\u0980' <= c <= '\u09FF')
        if bengali_chars > len(query) * 0.3:
            return 'bn'
        return 'en'


def is_mixed_language(query: str) -> bool:
    """Check if query mixes Bangla and English."""
    bengali_chars = sum(1 for c in query if '\u0980' <= c <= '\u09FF')
    english_chars = sum(1 for c in query if c.isascii() and c.isalpha())
    
    total_chars = len(query)
    if total_chars == 0:
        return False
    
    bengali_ratio = bengali_chars / total_chars
    english_ratio = english_chars / total_chars
    
    return bengali_ratio > 0.1 and english_ratio > 0.1

print("✅ Language detector loaded")

In [ ]:
# Step 3: Translator (MarianMT)
from transformers import MarianMTModel, MarianTokenizer

# Global models (loaded once)
_bn_to_en_model = None
_en_to_bn_model = None
_bn_to_en_tokenizer = None
_en_to_bn_tokenizer = None


def _load_bn_to_en():
    """Load Bangla to English translator."""
    global _bn_to_en_model, _bn_to_en_tokenizer
    if _bn_to_en_model is None:
        print("Loading Bangla→English model...")
        model_name = "Helsinki-NLP/Opus-MT-bn-en"
        _bn_to_en_tokenizer = MarianTokenizer.from_pretrained(model_name)
        _bn_to_en_model = MarianMTModel.from_pretrained(model_name)
    return _bn_to_en_model, _bn_to_en_tokenizer


def _load_en_to_bn():
    """Load English to Bangla translator."""
    global _en_to_bn_model, _en_to_bn_tokenizer
    if _en_to_bn_model is None:
        print("Loading English→Bangla model...")
        model_name = "Helsinki-NLP/Opus-MT-en-bn"
        _en_to_bn_tokenizer = MarianTokenizer.from_pretrained(model_name)
        _en_to_bn_model = MarianMTModel.from_pretrained(model_name)
    return _en_to_bn_model, _en_to_bn_tokenizer


def translate_bn_to_en(text: str) -> str:
    """Translate Bangla to English."""
    if not text or not text.strip():
        return ""
    
    try:
        model, tokenizer = _load_bn_to_en()
        inputs = tokenizer(text, return_tensors="pt", max_length=512, truncation=True)
        outputs = model.generate(**inputs)
        translated = tokenizer.decode(outputs[0], skip_special_tokens=True)
        return translated
    except Exception as e:
        print(f"Translation error: {e}")
        return text


def translate_en_to_bn(text: str) -> str:
    """Translate English to Bangla."""
    if not text or not text.strip():
        return ""
    
    try:
        model, tokenizer = _load_en_to_bn()
        inputs = tokenizer(text, return_tensors="pt", max_length=512, truncation=True)
        outputs = model.generate(**inputs)
        translated = tokenizer.decode(outputs[0], skip_special_tokens=True)
        return translated
    except Exception as e:
        print(f"Translation error: {e}")
        return text


def translate_query(text: str, source_lang: str, target_lang: str) -> str:
    """Translate query from source to target language."""
    if source_lang == target_lang:
        return text
    
    if source_lang == 'bn' and target_lang == 'en':
        return translate_bn_to_en(text)
    elif source_lang == 'en' and target_lang == 'bn':
        return translate_en_to_bn(text)
    else:
        return text

print("✅ Translator module loaded")

In [ ]:
# Step 4: Query Processor (Main Pipeline)
class QueryProcessor:
    """Process queries for cross-lingual search."""
    
    def __init__(self):
        self.lang_detected = None
        self.normalized = None
        self.original = None
        self.translated = None
    
    def process(self, query: str):
        """Process query through pipeline."""
        self.original = query
        
        # Step 1: Normalize
        self.normalized = self._normalize(query)
        
        # Step 2: Detect language
        self.lang_detected = detect_language(self.normalized)
        
        # Step 3: Translate to other language
        target_lang = 'en' if self.lang_detected == 'bn' else 'bn'
        self.translated = translate_query(
            self.normalized, 
            self.lang_detected, 
            target_lang
        )
        
        result = {
            'original': self.original,
            'language': self.lang_detected,
            'normalized': self.normalized,
            'translated': self.translated,
            'both_versions': [self.normalized, self.translated],
            'is_mixed': is_mixed_language(self.normalized),
        }
        
        return result
    
    def _normalize(self, text: str) -> str:
        """Normalize: trim, remove extra spaces."""
        if not text:
            return ""
        text = text.strip()
        text = ' '.join(text.split())
        return text


def process_query(query: str) -> dict:
    """Convenience function."""
    processor = QueryProcessor()
    return processor.process(query)

print("✅ Query processor loaded")

In [ ]:
# Step 5: Test the pipeline
print("\n" + "="*60)
print("MODULE B - Query Processing Tests")
print("="*60)

test_queries = [
    "Bangladesh politics",
    "বাংলাদেশের রাজনীতি",
    "Amir Khan actor",
    "আমির খান অভিনেতা",
    "education in dhaka",
    "ঢাকায় শিক্ষা ব্যবস্থা",
]

for query in test_queries:
    print(f"\n📝 Query: {query}")
    try:
        result = process_query(query)
        print(f"   Language: {result['language']}")
        print(f"   Normalized: {result['normalized']}")
        print(f"   Translated: {result['translated']}")
        print(f"   Mixed language: {result['is_mixed']}")
        print("   ✅ OK")
    except Exception as e:
        print(f"   ❌ Error: {e}")

print("\n" + "="*60)
print("✅ All tests complete!")
print("="*60)